In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_1samp, wilcoxon

DATA_PATH = "results/SD vs Mitsua (Responses) - Form Responses 1.csv"

# Names shown for the two anonymized options in the survey form.
MODEL_1 = "Juggernaut XL"
MODEL_2 = "Mitsua LoRA"

In [ ]:
df = pd.read_csv(DATA_PATH)

# The 21 comparison questions share one title; pandas suffixes repeats as .1, .2, ...
response_cols = [column for column in df.columns if column.startswith("Which image looks better?")]
assert len(response_cols) == 21, f"Expected 21 comparison questions, found {len(response_cols)}"
assert df[response_cols].isin(["Model 1", "Model 2"]).all().all(), "Unexpected response values"

print(f"Participants: {len(df)}")

In [ ]:
# Participant-level tests: each participant's share of choices favouring Model 2.
n_questions = len(response_cols)
model2_choices = (df[response_cols] == "Model 2").sum(axis=1)
pref_model2 = model2_choices / n_questions
print(f"Mean preference for {MODEL_2}: {pref_model2.mean():.3f}")

t_res = ttest_1samp(pref_model2, 0.5)
print(f"One-sample t-test vs 0.5: t = {t_res.statistic:.3f}, p = {t_res.pvalue:.3g}")

# Rank integer differences (2k - n, proportional to k/n - 0.5) so participants equally far
# from 50% tie exactly; float proportions such as 10/21 and 11/21 round differently and
# would silently break those ties, changing W and p.
w_res = wilcoxon(2 * model2_choices - n_questions)
print(f"Wilcoxon signed-rank test vs 0.5: W = {w_res.statistic:.1f}, p = {w_res.pvalue:.3g}")

In [ ]:
count_model1 = int((df[response_cols] == "Model 1").sum().sum())
count_model2 = int((df[response_cols] == "Model 2").sum().sum())
total_votes = count_model1 + count_model2

print(f"{MODEL_1}: {count_model1} votes")
print(f"{MODEL_2}: {count_model2} votes")
print(f"Total: {total_votes} votes")

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(8, 6))
plt.pie(
    [count_model1, count_model2],
    labels=[MODEL_1, MODEL_2],
    autopct="%1.1f%%",
    colors=sns.color_palette("Set2"),
)
plt.title(
    "Global Convenience Retailer Marketing Campaign\n"
    "Preference between Web-Scale and Rights-Cleared Models"
)
plt.tight_layout()
plt.savefig("results/SOTAvsPublic.png", format="png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
models = [MODEL_1, MODEL_2]
counts = [count_model1, count_model2]

plt.figure(figsize=(8, 6))
sns.barplot(x=models, y=counts, hue=models, palette="Set2", legend=False)
plt.title(f"Preference Counts for {MODEL_1} vs {MODEL_2}", fontsize=16)
plt.xlabel("Model", fontsize=14)
plt.ylabel("Number of Votes", fontsize=14)

for index, count in enumerate(counts):
    plt.text(index, count + 50, str(count), ha="center", fontsize=12)

plt.tight_layout()
plt.show()